# Spotify Popularity Predictor


1. Load the dataset from Kaggle
2. Clean the data
3. Define the target (binary: popular vs not)
4. Explore the data (EDA)
5. Prepare features (split + scale)
6. Build a Deep Neural Network with Keras
7. Train the model
8. Evaluate on test data
9. Save the model for deployment


## 1. Setup — Install and Import Libraries

In [1]:
!pip install kagglehub --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("TensorFlow version:", tf.__version__)

c:\Users\frede\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'pandas'

## 2. Load Data
Using `kagglehub` to download directly. 

In [ ]:
path = kagglehub.dataset_download("fcpercival/160k-spotify-songs-sorted")
print("Path:", path)

In [ ]:
import os
df = pd.read_csv(os.path.join(path, "data.csv"))
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Inspect the Data

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

## 4. Clean the Data

Steps:
1. Drop duplicates (same song + same artist)
2. Remove non-music entries (zero tempo = sleep sounds)
3. Drop any remaining missing values

In [ ]:
print(f"Rows before cleaning: {len(df)}")

df = df.drop_duplicates(subset=['name', 'artists'])
df = df[(df['tempo'] > 0) & (df['danceability'] > 0)]
df = df.dropna().reset_index(drop=True)

print(f"Rows after cleaning:  {len(df)}")

## 5. Define the Target

We define a song as **popular** if its popularity score is ≥ 50 (Spotify popularity is 0–100).


In [ ]:
df['is_popular'] = (df['popularity'] >= 50).astype(int)

print("Class distribution:")
print(df['is_popular'].value_counts())
print(f"\n% popular: {df['is_popular'].mean() * 100:.1f}%")

In [ ]:
plt.figure(figsize=(6, 4))
df['is_popular'].value_counts().plot(kind='bar', color=['steelblue', 'coral'])
plt.title("Class Distribution: Popular vs Not Popular")
plt.xlabel("is_popular (0 = not, 1 = popular)")
plt.ylabel("Number of songs")
plt.xticks(rotation=0)
plt.show()

## 6. Exploratory Data Analysis


### 6.1 Distribution of audio features

In [ ]:
audio_features = [
    'acousticness', 'danceability', 'energy', 'instrumentalness',
    'liveness', 'loudness', 'speechiness', 'tempo', 'valence'
]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(audio_features):
    axes[i].hist(df[feature], bins=50, color='steelblue', edgecolor='black')
    axes[i].set_title(feature)

plt.suptitle("Distribution of Audio Features", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 6.2 Correlation between features

In [ ]:
corr = df[audio_features + ['popularity']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title("Correlation Between Audio Features and Popularity")
plt.show()

### 6.3 Audio features by popularity


In [ ]:
# Compare mean feature values for popular vs not popular
comparison = df.groupby('is_popular')[audio_features].mean().T
comparison.columns = ['Not Popular', 'Popular']
comparison['Difference'] = comparison['Popular'] - comparison['Not Popular']

plt.figure(figsize=(10, 6))
comparison['Difference'].plot(kind='barh', color='teal')
plt.title("Difference in Average Audio Features (Popular − Not Popular)")
plt.xlabel("Difference (positive = higher in popular songs)")
plt.axvline(0, color='black', linewidth=0.5)
plt.show()

comparison

## 7. Prepare Features for the Neural Network

- **Train/test split:** 80% / 20%, stratified so class balance is preserved
- **Scaling:** `StandardScaler` (mean=0, std=1) — important for NN training stability

In [ ]:
X = df[audio_features].values
y = df['is_popular'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set:     {X_test_scaled.shape}")
print(f"\nClass balance in training: {y_train.mean()*100:.1f}% popular")
print(f"Class balance in test:     {y_test.mean()*100:.1f}% popular")

## 8. Build the Deep Neural Network


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(9,)),
    layers.Dense(64, activation='relu', name='hidden_1'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu', name='hidden_2'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu', name='hidden_3'),
    layers.Dense(1, activation='sigmoid', name='output')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 9. Train the Model

- **Epochs:** 40 (one epoch = one full pass through training data)
- **Batch size:** 256 (samples processed before weight update)
- **Validation split:** 20% of training set used to monitor overfitting



In [ ]:
# Stop early if validation loss stops improving
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=40,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

## 10. Training Curves



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Training loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation loss', linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss over epochs")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Training accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation accuracy', linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy over epochs")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Evaluate on Test Set


In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
y_pred_prob = model.predict(X_test_scaled, verbose=0).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['Not Popular', 'Popular']))

### 11.1 Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Popular', 'Popular'],
            yticklabels=['Not Popular', 'Popular'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Popularity Prediction')
plt.show()

### 11.2 ROC Curve
AUC (Area Under Curve) of 1.0 = perfect classifier, 0.5 = random guessing.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'DNN (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 12. Test the Model on Real Songs


In [ ]:
# Pick songs from the original (unscaled) test data
test_df = df.iloc[X_test.shape[0]:].copy()  
sample = df.sample(8, random_state=42)
sample_X = scaler.transform(sample[audio_features].values)
sample_pred_prob = model.predict(sample_X, verbose=0).flatten()

results = sample[['name', 'artists', 'popularity', 'is_popular']].copy()
results['predicted_prob'] = sample_pred_prob.round(2)
results['prediction'] = (sample_pred_prob > 0.5).astype(int)
results['correct'] = (results['prediction'] == results['is_popular']).map({True: '✅', False: '❌'})

results.reset_index(drop=True)

## 13. Save the Model for Deployment


In [ ]:
# Save the model and scaler
model.save('popularity_model.keras')

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Saved:")
print("  - popularity_model.keras")
print("  - scaler.pkl")

In [ ]:
# Download files from Colab
from google.colab import files

files.download('popularity_model.keras')
files.download('scaler.pkl')